Starting Day 1 on 5/25/2026 

In [10]:
import matplotlib.pyplot as plt

In [2]:
words = open('names.txt', 'r').read().splitlines()


In [8]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [4]:
print(min(len(w) for w in words))
print(max(len(w) for w in words))

2
15


First we build the Bigram language model. We will be working with a list of names, and we want to learn how to generate new names that look similar to the ones in the list.

In [3]:
b = {}
for w in words[:]:
    chs = ['<S'] + list(w) + ['<E']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1
# To figure out how common the characters are next to each other, we are going to count!

In [22]:
# sorted(b.items(), key= lambda kv: -kv[1]) 
# It's going to be convenient for us to use a 2D array instead of a dict

In [4]:
import torch
a = torch.zeros((3, 5), dtype= torch.int32) # since we will only use counts, int is better
a

tensor([[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]], dtype=torch.int32)

In [7]:
# we require a bigger array, that contains all alphabets + two new characters (<S and <E)
N = torch.zeros((28, 28), dtype = torch.int32)

We need a way to convert the characters to numbers

In [5]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i for i, s in enumerate(chars)} # enumerate gives index and the element in that index
stoi['<S'] = 26
stoi['<E'] = 27

In [8]:
for w in words[:]:
    chs = ['<S'] + list(w) + ['<E']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

In [53]:
N[3, 3].item() # gives the direct integer instead of a tensor

149

In [ ]:
itos = {i:s for s, i in stoi.items()}
plt.figure(figsize= (16, 16))
plt.imshow(N, cmap= 'Blues')
for i in range(28):
    for j in range(28):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha='center', va='bottom', color= 'blue')
        plt.text(j, i, N[i,j].item(), ha='center', va='top', color= 'red') 
plt.axis('off')

Observe carefully to see that we have a row of 0s where E being the starting char, which never happens and same applied to S being the ending char. Next, we have to tidy up this matrix.

Done Day 1 on 5/25/2026 at 19:53 -> video time (22:10)

Starting Day 2 on 5/26/2026 at 17:52

(28, 28) matrix will be changed to (27, 27) to remove the extra unused character.

In [12]:
N = torch.zeros((27, 27), dtype = torch.int32)

In [13]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)} # enumerate gives index and the element in that index
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
# itos

In [14]:
for w in words[:]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

In [ ]:
itos = {i:s for s, i in stoi.items()}
plt.figure(figsize= (16, 16))
plt.imshow(N, cmap= 'Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha='center', va='bottom', color= 'blue')
        plt.text(j, i, N[i,j].item(), ha='center', va='top', color= 'red') 
plt.axis('off')

In [15]:
N[0]

tensor([   0, 4410, 1306, 1542, 1690, 1531,  417,  669,  874,  591, 2422, 2963,
        1572, 2538, 1146,  394,  515,   92, 1639, 2055, 1308,   78,  376,  307,
         134,  535,  929], dtype=torch.int32)

We want to convert these counts to probabilities to sample them

In [16]:
p = N[0].float() # convert to float for division
p = p / p.sum() # normalize to get probabilities
p

tensor([0.0000, 0.1377, 0.0408, 0.0481, 0.0528, 0.0478, 0.0130, 0.0209, 0.0273,
        0.0184, 0.0756, 0.0925, 0.0491, 0.0792, 0.0358, 0.0123, 0.0161, 0.0029,
        0.0512, 0.0642, 0.0408, 0.0024, 0.0117, 0.0096, 0.0042, 0.0167, 0.0290])

In [17]:
g = torch.Generator().manual_seed(2147483647) # set the seed for reproducibility
ix = torch.multinomial(p, num_samples = 1, replacement = True, generator=g).item()
itos[ix]

'c'

In [18]:
p.sum() # the sum of probabilities should be 1 

tensor(1.)

Finishd Day 2 on 5/26/2026 at 18:06 -> video time (27:50)

Starting Day 3 on 5/27/2026 at 18:00

In [19]:
g = torch.Generator().manual_seed(2147483647) # set the seed for reproducibility
p = torch.rand(3, generator=g)
p = p/p.sum()
p

tensor([0.6064, 0.3033, 0.0903])

In [20]:
torch.multinomial(p, num_samples = 3, replacement = True) # gives us 20 samples from the distribution p, with replacement

tensor([0, 1, 0])

Something to notice here is that, we are sampling from the distribution p, and carefully looking at it's output, you have a 60% chance of getting a 0, 30 % for 1, and 10% for 2. And that's what happens in multinomial sampling!

In [21]:
g = torch.Generator().manual_seed(2147483647) # set the seed for reproducibility
for i in range(20):
    
    out = []
    ix = 0
    while True:
        p = N[ix].float()
        p = p/ p.sum()
        ix = torch.multinomial(p, num_samples = 1, replacement = True, generator=g).item()
        out.append(itos[ix]) 
        if ix == 0:
            break 
    print("".join(out))

cexze.
momasurailezitynn.
konimittain.
llayn.
ka.
da.
staiyaubrtthrigotai.
moliellavo.
ke.
teda.
ka.
emimmsade.
enkaviyny.
ftlspihinivenvorhlasu.
dsor.
br.
jol.
pen.
aisan.
ja.


Finished Day 3 on 5/27/2026 at 18:34 -> video time (33:24)

Starting Day 4 on 5/28/2026 at 12:00

In [22]:
P = N.float()
P.sum() # If you sum it this way, it's adding up all the count across the N matrix, but we want to add values across the row
# because we know that their sum of probabilities will equal 1. 

tensor(228146.)

In [29]:
P.sum(0, keepdim= True).shape # this sums up values across the columns, so you get a row vector

torch.Size([1, 27])

In [ ]:
P.sum(1, keepdim= True).shape # this is what we want
# but why are we saying keepdim as true? 
# It's because if it's false, it would squeeze the array to one dim. For example, 27 X 1, would become just 27.

torch.Size([27, 1])

In [ ]:
P.sum(1).shape
# This would simply be 27 because keepdim is false.
# And when we do (27, 27) with (27) - internally it would go as
# 27, 27
#   , 27 -> (1, 27)
# so instead of column vector, it will be a row vector(1 X 27) and because of that, it would normalize across columns instead of rows like we intend to

torch.Size([27])

A fun fact - The counts sum across the rows and columns is identical

In [34]:
P = (N+1).float()
P_sum = P.sum(1, keepdim= True)
# Is it possible to divide these terms? (27, 27) matrix with (27, 1). Yes it is because of broadcasting!
P /= P_sum 
P.shape

torch.Size([27, 27])

In [24]:
P[0].sum()

tensor(1.)

In [35]:
g = torch.Generator().manual_seed(2147483647) # set the seed for reproducibility
for i in range(5):
    
    out = []
    ix = 0
    while True:
        
        p = P[ix]
        #p = N[ix].float()
        #p = p/ p.sum()
        ix = torch.multinomial(p, num_samples = 1, replacement = True, generator=g).item()
        out.append(itos[ix]) 
        if ix == 0:
            break 
    print("".join(out))

cexze.
momasurailezitynn.
konimittain.
llayn.
ka.


Finished Day 4 on 5/28/2026 at 12:50 -> video time (50:22)

Starting Day 5 on 5/29/2026 at 18:37

In [ ]:
for w in words[:3]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2] 
        print(f'{ch1}{ch2}: {prob:.4f}')

MLE says that the likelihood of the entire data is given by product of individual probabilites(which defines model quality)
So we can calculate the likelihood of the data by multiplying the probabilities of each bigram in the data.
But this is a very small number, so we can take the log of the likelihood to get a more manageable number. 
The log of the likelihood is given by the sum of the log of the probabilities of each bigram in the data.

In [ ]:
log_likelihood = 0.0
n= 0
for w in words[:3]:
# for w in ["yaswanthqp"]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2] 
        logprob = torch.log(prob)
        log_likelihood += logprob
        n += 1
        print(f'{ch1}{ch2}: {prob:.4f}')
        
print(f'log likelihood: {log_likelihood:.4f}')
# take negative log -> -log is convex, and we can minimize it but log is concave and we will maximize it. Loss is dealt in minimizing. 
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}') # average negative log likelihood per bigram, this is the loss that we want to minimize.

.e: 0.0478
em: 0.0377
mm: 0.0253
ma: 0.3885
a.: 0.1958
.o: 0.0123
ol: 0.0779
li: 0.1774
iv: 0.0152
vi: 0.3508
ia: 0.1380
a.: 0.1958
.a: 0.1376
av: 0.0246
va: 0.2473
a.: 0.1958
log likelihood: -38.8086
nll=tensor(38.8086)
2.4255354404449463


Normally, this would give -inf (the one with my name), but since we have added 1 to the counts, we can avoid that. This is called Laplace smoothing, and it is a common technique to avoid zero probabilities in language models.

Finished Day 5 on 5/29/2026 at 19:02 -> video time (58:21)

Starting Day 6 on 5/30/2026 at 16:25

Goal: Maxmize the likelihood(product of the probabilites) of the data w.r.t model parameters (statistical modelling)
Which is equivalent to maximizing the log likelihood (because log is monotonic) 
Equivalent to minimizing the negative log likelihood (because we want to minimize the loss)
Equivalent to minimizing average negative log likelihood

Now we are moving to building a Neural net.

In [39]:
# construct a training set
xs, ys = [], []

for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2] 
        print(ch1, ch2)
        xs.append(ix1)
        ys.append(ix2)
        
xs = torch.tensor(xs)
ys = torch.tensor(ys)

. e
e m
m m
m a
a .


In [41]:
xs, ys
# explanation: For 0 input, you'd want 5 (means 'e'), For 5 input, you want 13 ('m') and so on.
# These are indices and you will give them to one hot encoder, to get 1s wherever the index is, and you convert that to floats because NN requires floats.

(tensor([ 0,  5, 13, 13,  1]), tensor([ 5, 13, 13,  1,  0]))

In [42]:
import torch.nn.functional as F
xenc = F.one_hot(xs, num_classes = 27).float()

In [44]:
xenc.dtype, xenc.shape

(torch.float32, torch.Size([5, 27]))

Finished Day 6 on 5/30/2026 at 17:31 -> video time (1:13:59)